Task 3 — Individual and ensemble prediction models
Investigate the contribution of different feature groups
Perform an ablation study to determine which types of information contribute to predictive performance.
At a minimum, compare:
1. contextual and weather variables;
2. contextual, weather, and tariff variables;
3. the previous variables plus charging-post history;
4. the previous variables plus charging-post and user history.
The purpose is not merely to identify the combination with the highest score. You must analyse:
– whether the tariff provides information beyond time-of-day variables;
– how much charging-post history contributes;
– whether user history produces an additional benefit;
– whether the effect is consistent across temporal periods;
– whether particular feature groups introduce operational or ethical concerns.
Groups may propose additional leakage-safe features, provided their construction and availability at prediction
time are clearly justified.
Develop individual classification models
Select and evaluate classifiers representing at least three different modelling principles:

– linear classifiers;
– probabilistic classifiers;
– instance-based methods;
– decision trees;
– kernel-based methods.

The choice of algorithms is the responsibility of each group. It must be justified in relation to:
– data types;
– class imbalance;
– model assumptions;
– interpretability;
– computational requirements;
– expected relationships between predictors and the target.
Using many algorithms without analysis is not considered preferable to a smaller, well-justified comparison.
Develop ensemble models
Evaluate at least three ensemble approaches representing different ensemble principles.
Possible approaches include:
– bagging;
– random forests;
– boosting;
– soft or hard voting;
– stacking.
If stacking is used, the predictions used to train the meta-model must be generated without leakage. Standard
random internal cross-validation is not appropriate when it allows future observations to influence predictions
for earlier observations.
Explain why each ensemble could reasonably improve upon its constituent models.


In [2]:
import pandas as pd
from sklearn.model_selection import TimeSeriesSplit
import numpy as np
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import make_pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OneHotEncoder, StandardScaler, FunctionTransformer
from sklearn.metrics import average_precision_score
from sklearn.ensemble import HistGradientBoostingClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.naive_bayes import GaussianNB
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier, VotingClassifier

import sklearn
from sklearn.base import BaseEstimator, ClassifierMixin
from sklearn.naive_bayes import CategoricalNB
from sklearn.preprocessing import OrdinalEncoder
from scipy.special import logsumexp

## 1. Preparação

In [3]:
file_path = r"C:\Users\Raphael\Desktop\Mineração de Dados\Minera-o_de_Dados_MEI\data\Charging_Data_tratado.csv"
df = pd.read_csv(file_path)

df.shape

FileNotFoundError: [Errno 2] No such file or directory: 'C:\\Users\\Raphael\\Desktop\\Mineração de Dados\\Minera-o_de_Dados_MEI\\data\\Charging_Data_tratado.csv'

In [ ]:
df['Start Time'] = pd.to_datetime(df['Start Time'])

df['Start Time'].dtype

dtype('<M8[us]')

In [ ]:
df = df.sort_values('Start Time').reset_index(drop=True)

df['Start Time'].head()

0   2019-12-31 21:27:03
1   2019-12-31 21:50:56
2   2019-12-31 22:12:46
3   2019-12-31 22:38:55
4   2019-12-31 23:06:44
Name: Start Time, dtype: datetime64[us]

In [ ]:
# Hora de início da sessão (0 a 23), já conhecida no momento da previsão
df['hora'] = df['Start Time'].dt.hour

df['hora'].value_counts().sort_index()

hora
0     11381
1      7012
2      5101
3      4548
4      3969
5      3876
6      5670
7      7076
8      9746
9     17125
10    22106
11    30205
12    31555
13    30480
14    31632
15    31779
16    27663
17    21716
18    23848
19    26068
20    21974
21    22843
22    24104
23    19600
Name: count, dtype: int64

In [ ]:
# Dia da semana (0 = segunda-feira, 6 = domingo)
df['dia_da_semana'] = df['Start Time'].dt.dayofweek

df['dia_da_semana'].value_counts().sort_index()

dia_da_semana
0    62064
1    61465
2    62093
3    63220
4    65562
5    64463
6    62210
Name: count, dtype: int64

In [ ]:
# split do df (80-20).
def dividir_data(df, n_folds, prop_validacao, tamanho_treino):
    corte = int(len(df) * 0.8)
    treino = df.iloc[:corte]
    test = df.iloc[corte:]

    tamanho_validacao = int(len(treino) * prop_validacao / n_folds)
    tss = TimeSeriesSplit(n_splits=n_folds, test_size=tamanho_validacao, max_train_size=tamanho_treino)

    folds = []
    for indices_treino, indices_validacao in tss.split(treino):
        folds.append((treino.iloc[indices_treino], treino.iloc[indices_validacao]))

    return folds, test

In [ ]:
# 80% treino e 20% teste, com 3 folds feitos só no treino
# as validações ocupam os últimos 45% do treino e cada treino tem 180 000 sessões
# o teste fica guardado e não é usado nesta task
folds, test = dividir_data(df, 3, 0.45, 180000)
test.shape

(88216, 34)

In [ ]:
treino_1 = folds[0][0]
treino_1.shape

(180000, 34)

In [ ]:
validacao_1 = folds[0][1]
validacao_1.shape

(52929, 34)

In [ ]:
treino_2 = folds[1][0]
treino_2.shape

(180000, 34)

In [ ]:
validacao_2 = folds[1][1]
validacao_2.shape

(52929, 34)

In [ ]:
treino_3 = folds[2][0]
treino_3.shape

(180000, 34)

In [ ]:
validacao_3 = folds[2][1]
validacao_3.shape

(52929, 34)

### Baselines por fold

In [ ]:
# AUC-PR do baseline de regra (taxa de falhas do posto no dia anterior)
regra_1 = average_precision_score(validacao_1['is_Abnormal'], validacao_1['post_previous_abnormal_rate_1D'])
regra_1

0.4503839848595308

In [ ]:
regra_2 = average_precision_score(validacao_2['is_Abnormal'], validacao_2['post_previous_abnormal_rate_1D'])
regra_2

0.602264671977131

In [ ]:
regra_3 = average_precision_score(validacao_3['is_Abnormal'], validacao_3['post_previous_abnormal_rate_1D'])
regra_3

0.5901262937508409

## 2. Grupos de variáveis

In [ ]:
# A tarifa depende só da hora: em cada hora aparece sempre o mesmo período tarifário
pd.crosstab(df['hora'], df['tariff_period'])

tariff_period,off_peak_00_08,off_peak_11_13,off_peak_22_24,peak_08_11,peak_13_19,peak_21_22,sharp_19_21
hora,,,,,,,
0,11381,0,0,0,0,0,0
1,7012,0,0,0,0,0,0
2,5101,0,0,0,0,0,0
3,4548,0,0,0,0,0,0
4,3969,0,0,0,0,0,0
5,3876,0,0,0,0,0,0
6,5670,0,0,0,0,0,0
7,7076,0,0,0,0,0,0
8,0,0,0,9746,0,0,0


In [ ]:
# Número de períodos tarifários diferentes em cada hora
df.groupby('hora')['tariff_period'].nunique()

hora
0     1
1     1
2     1
3     1
4     1
5     1
6     1
7     1
8     1
9     1
10    1
11    1
12    1
13    1
14    1
15    1
16    1
17    1
18    1
19    1
20    1
21    1
22    1
23    1
Name: tariff_period, dtype: int64

In [ ]:
# G1: contexto e clima
grupo_1 = ['Location Information', 'District Name', 'hora', 'dia_da_semana',
           'Temperature(℃)', 'Relative Humidity(%)', 'Precipitation(mm)']

In [ ]:
# G2: G1 + tarifa
grupo_2 = grupo_1 + ['tariff_period']
print(grupo_2)

['Location Information', 'District Name', 'hora', 'dia_da_semana', 'Temperature(℃)', 'Relative Humidity(%)', 'Precipitation(mm)', 'tariff_period']


In [ ]:
# Histórico do posto
colunas_posto = ['post_previous_sessions_1D', 'post_previous_abnormal_1D', 'post_previous_abnormal_rate_1D',
                 'post_previous_sessions_7D', 'post_previous_abnormal_7D', 'post_previous_abnormal_rate_7D',
                 'post_previous_abnormal_30D', 'post_previous_abnormal_rate_30D',
                 'post_hours_since_previous_completed_session', 'post_no_previous_completed_session',
                 'post_hours_since_previous_completed_abnormal', 'post_no_previous_completed_abnormal']
len(colunas_posto)

12

In [ ]:
# G3: G2 + histórico do posto
grupo_3 = grupo_2 + colunas_posto
len(grupo_3)

20

In [ ]:
# Histórico do usuário
# a user_active_sessions_at_start entra aqui porque descreve o estado do usuário (é ela que mostra as frotas)
colunas_usuario = ['user_previous_sessions_30D', 'user_previous_abnormal_rate_30D',
                   'user_previous_sessions_180D', 'user_previous_abnormal_rate_180D',
                   'user_previous_sessions_730D',
                   'user_days_since_previous_completed_session', 'user_no_previous_completed_session',
                   'user_days_since_previous_completed_abnormal', 'user_no_previous_completed_abnormal',
                   'user_active_sessions_at_start']
len(colunas_usuario)

10

In [ ]:
# G4: G3 + histórico do usuário
grupo_4 = grupo_3 + colunas_usuario
print(grupo_4)

['Location Information', 'District Name', 'hora', 'dia_da_semana', 'Temperature(℃)', 'Relative Humidity(%)', 'Precipitation(mm)', 'tariff_period', 'post_previous_sessions_1D', 'post_previous_abnormal_1D', 'post_previous_abnormal_rate_1D', 'post_previous_sessions_7D', 'post_previous_abnormal_7D', 'post_previous_abnormal_rate_7D', 'post_previous_abnormal_30D', 'post_previous_abnormal_rate_30D', 'post_hours_since_previous_completed_session', 'post_no_previous_completed_session', 'post_hours_since_previous_completed_abnormal', 'post_no_previous_completed_abnormal', 'user_previous_sessions_30D', 'user_previous_abnormal_rate_30D', 'user_previous_sessions_180D', 'user_previous_abnormal_rate_180D', 'user_previous_sessions_730D', 'user_days_since_previous_completed_session', 'user_no_previous_completed_session', 'user_days_since_previous_completed_abnormal', 'user_no_previous_completed_abnormal', 'user_active_sessions_at_start']


## 3. Pré-processamento

In [ ]:
# Variáveis categóricas: one-hot nas duas versões
colunas_categoricas = ['Location Information', 'District Name', 'tariff_period']

In [ ]:
# Hora e dia da semana: one-hot nos modelos lineares (o risco não é linear na hora) e número nas árvores
tempo = ['hora', 'dia_da_semana']

In [ ]:
# Contagens e tempos com skew acima de 3 (Task 1): log1p nos modelos lineares
log_data = ['post_hours_since_previous_completed_session', 'post_hours_since_previous_completed_abnormal',
               'post_previous_abnormal_1D', 'post_previous_abnormal_7D', 'post_previous_abnormal_30D',
               'user_previous_sessions_30D', 'user_previous_sessions_180D', 'user_previous_sessions_730D',
               'user_days_since_previous_completed_session', 'user_days_since_previous_completed_abnormal',
               'user_active_sessions_at_start']


In [ ]:
# Monta o pré-processamento só com as colunas que existem no grupo
# linear=True: regressão logística e Naive Bayes (sem NaN, com log1p e escala)
# linear=False: árvores (os NaN ficam, sem log1p nem escala)
def criar_prep(colunas, linear):
    categoricas = [c for c in colunas if c in colunas_categoricas]
    tempo_prep = [c for c in colunas if c in tempo]
    com_log = [c for c in colunas if c in log_data]
    outras = [c for c in colunas if c not in categoricas + tempo + com_log]

    one_hot = OneHotEncoder(handle_unknown='ignore', sparse_output=False)

    if linear:
        # NaN -> máximo do treino ("nunca aconteceu" = valor alto; a flag *_no_previous_* continua a dizer que é "nunca"), depois log1p e escala
        log = make_pipeline(SimpleImputer(strategy=np.max),
                            FunctionTransformer(np.log1p, feature_names_out='one-to-one'),
                            StandardScaler())
        return ColumnTransformer([
            ('categoricas', one_hot, categoricas + tempo_prep),
            ('log', log, com_log),
            ('numericas', StandardScaler(), outras),
        ])
    else:
        # hora, dia_da_semana e as numéricas passam como estão
        return ColumnTransformer([
            ('categoricas', one_hot, categoricas),
        ], remainder='passthrough')

## 4. Função de avaliação

In [ ]:
# AUC-PR da regra em cada fold, para comparar com os modelos
regras = [regra_1, regra_2, regra_3]
print(regras)

[0.4503839848595308, 0.602264671977131, 0.5901262937508409]


In [ ]:
# Taxa de anomalia de cada fold (= AUC-PR do no-skill)
taxas = [validacao_1['is_Abnormal'].mean(), validacao_2['is_Abnormal'].mean(), validacao_3['is_Abnormal'].mean()]
taxas

[np.float64(0.15790965255342063),
 np.float64(0.19679192881029303),
 np.float64(0.21035727106123298)]

In [ ]:
# Treina o modelo no treino de cada fold e devolve a AUC-PR de cada validação
# como o pré-processamento está dentro do modelo (pipeline), ele também é ajustado só no treino de cada fold
def avaliar(modelo, colunas):
    resultados = []
    for treino, validacao in folds:
        modelo.fit(treino[colunas], treino['is_Abnormal'])
        prob = modelo.predict_proba(validacao[colunas])[:, 1]
        resultados.append(average_precision_score(validacao['is_Abnormal'], prob))
    return resultados

In [ ]:
# Resumo de um modelo, como definido no protocolo da Task 2:
# valor de cada fold, média simples, desvio-padrão, pior fold e em quantos folds passa da regra
def resumir(resultados):
    folds_ganhos = 0
    for i in range(3):
        if resultados[i] > regras[i]:
            folds_ganhos = folds_ganhos + 1
    return pd.Series({'fold 1': resultados[0], 'fold 2': resultados[1], 'fold 3': resultados[2],
                      'média': pd.Series(resultados).mean(),
                      'desvio': pd.Series(resultados).std(),
                      'pior fold': min(resultados),
                      'folds > regra': folds_ganhos})

## 5. Ablação

### Regressão logística

In [ ]:
# G1: contexto e clima
modelo_lr_g1 = make_pipeline(criar_prep(grupo_1, True), LogisticRegression(class_weight='balanced', max_iter=1000))
resultados_lr_g1 = avaliar(modelo_lr_g1, grupo_1)
resultados_lr_g1

[0.24902410177233109, 0.31011679350546995, 0.33449125149561815]

In [ ]:
# G2: G1 + tarifa
modelo_lr_g2 = make_pipeline(criar_prep(grupo_2, True), LogisticRegression(class_weight='balanced', max_iter=1000))
resultados_lr_g2 = avaliar(modelo_lr_g2, grupo_2)
resultados_lr_g2

[0.2490554076672104, 0.31015648950676583, 0.3346312203753496]

In [ ]:
# G3: G2 + histórico do posto
modelo_lr_g3 = make_pipeline(criar_prep(grupo_3, True), LogisticRegression(class_weight='balanced', max_iter=1000))
resultados_lr_g3 = avaliar(modelo_lr_g3, grupo_3)
resultados_lr_g3

[0.5062081136939464, 0.6364413558112486, 0.6168249626874805]

In [ ]:
# G4: G3 + histórico do usuário
modelo_lr_g4 = make_pipeline(criar_prep(grupo_4, True), LogisticRegression(class_weight='balanced', max_iter=1000))
resultados_lr_g4 = avaliar(modelo_lr_g4, grupo_4)
resultados_lr_g4

[0.5320325514569119, 0.6522901018035159, 0.6272623154631753]

In [ ]:
ablacao_lr = pd.DataFrame({'G1': resumir(resultados_lr_g1), 'G2': resumir(resultados_lr_g2),
                           'G3': resumir(resultados_lr_g3), 'G4': resumir(resultados_lr_g4)}).T
ablacao_lr

,fold 1,fold 2,fold 3,média,desvio,pior fold,folds > regra
G1,0.249024,0.310117,0.334491,0.297877,0.044029,0.249024,0.0
G2,0.249055,0.310156,0.334631,0.297948,0.044075,0.249055,0.0
G3,0.506208,0.636441,0.616825,0.586491,0.070216,0.506208,3.0
G4,0.532033,0.652290,0.627262,0.603862,0.063452,0.532033,3.0


In [ ]:
# Ganho de cada grupo em relação ao anterior, fold a fold (G2 - G1, G3 - G2, G4 - G3)
ablacao_lr[['fold 1', 'fold 2', 'fold 3', 'média']].diff()

,fold 1,fold 2,fold 3,média
G1,NaN,NaN,NaN,NaN
G2,0.000031,0.000040,0.000140,0.000070
G3,0.257153,0.326285,0.282194,0.288544
G4,0.025824,0.015849,0.010437,0.017370


In [ ]:
# AUC-PR a dividir pela taxa de anomalia de cada fold (quantas vezes melhor que o no-skill)
ablacao_lr[['fold 1', 'fold 2', 'fold 3']] / taxas

,fold 1,fold 2,fold 3
G1,1.577004,1.575861,1.590110
G2,1.577202,1.576063,1.590776
G3,3.205682,3.234083,2.932273
G4,3.369221,3.314618,2.981890


### HistGradientBoosting

In [ ]:
# G1: contexto e clima
# early_stopping=False porque, por defeito, ele separa 10% aleatório do treino para decidir quando parar
modelo_hgb_g1 = make_pipeline(criar_prep(grupo_1, False), HistGradientBoostingClassifier(class_weight='balanced', early_stopping=False, random_state=42))
resultados_hgb_g1 = avaliar(modelo_hgb_g1, grupo_1)
resultados_hgb_g1

[0.23247281167472733, 0.300440421716687, 0.3420654564996465]

In [ ]:
# G2: G1 + tarifa
modelo_hgb_g2 = make_pipeline(criar_prep(grupo_2, False), HistGradientBoostingClassifier(class_weight='balanced', early_stopping=False, random_state=42))
resultados_hgb_g2 = avaliar(modelo_hgb_g2, grupo_2)
resultados_hgb_g2

[0.23294589922485148, 0.3020242097601176, 0.34656522157988534]

In [ ]:
# G3: G2 + histórico do posto
modelo_hgb_g3 = make_pipeline(criar_prep(grupo_3, False), HistGradientBoostingClassifier(class_weight='balanced', early_stopping=False, random_state=42))
resultados_hgb_g3 = avaliar(modelo_hgb_g3, grupo_3)
resultados_hgb_g3

[0.5377748582737041, 0.6607266327853737, 0.641610683433222]

In [ ]:
# G4: G3 + histórico do usuário
modelo_hgb_g4 = make_pipeline(criar_prep(grupo_4, False), HistGradientBoostingClassifier(class_weight='balanced', early_stopping=False, random_state=42))
resultados_hgb_g4 = avaliar(modelo_hgb_g4, grupo_4)
resultados_hgb_g4

[0.5635541625026558, 0.679380945871341, 0.6530067620251055]

In [ ]:
ablacao_hgb = pd.DataFrame({'G1': resumir(resultados_hgb_g1), 'G2': resumir(resultados_hgb_g2),
                            'G3': resumir(resultados_hgb_g3), 'G4': resumir(resultados_hgb_g4)}).T
ablacao_hgb

,fold 1,fold 2,fold 3,média,desvio,pior fold,folds > regra
G1,0.232473,0.300440,0.342065,0.291660,0.055321,0.232473,0.0
G2,0.232946,0.302024,0.346565,0.293845,0.057250,0.232946,0.0
G3,0.537775,0.660727,0.641611,0.613371,0.066162,0.537775,3.0
G4,0.563554,0.679381,0.653007,0.631981,0.060709,0.563554,3.0


In [ ]:
# Ganho de cada grupo em relação ao anterior, fold a fold
ablacao_hgb[['fold 1', 'fold 2', 'fold 3', 'média']].diff()

,fold 1,fold 2,fold 3,média
G1,NaN,NaN,NaN,NaN
G2,0.000473,0.001584,0.004500,0.002186
G3,0.304829,0.358702,0.295045,0.319526
G4,0.025779,0.018654,0.011396,0.018610


In [ ]:
# AUC-PR a dividir pela taxa de anomalia de cada fold
ablacao_hgb[['fold 1', 'fold 2', 'fold 3']] / taxas

,fold 1,fold 2,fold 3
G1,1.472189,1.526691,1.626117
G2,1.475185,1.534739,1.647508
G3,3.405586,3.357488,3.050100
G4,3.568839,3.452281,3.104275


## 5.1 Escolha da profundidade da árvore (G3)

In [ ]:
# Igual à avaliar(), mas devolve também a AUC-PR no próprio treino de cada fold (treino vs. validação)
def avaliar_treino_validacao(modelo, colunas):
    resultados_treino = []
    resultados_validacao = []
    for treino, validacao in folds:
        modelo.fit(treino[colunas], treino['is_Abnormal'])
        prob_treino = modelo.predict_proba(treino[colunas])[:, 1]
        prob_validacao = modelo.predict_proba(validacao[colunas])[:, 1]
        resultados_treino.append(average_precision_score(treino['is_Abnormal'], prob_treino))
        resultados_validacao.append(average_precision_score(validacao['is_Abnormal'], prob_validacao))
    return resultados_treino, resultados_validacao

In [ ]:
# Árvore de decisão com várias profundidades (mesmo pré-processamento das árvores)
profundidades = [3, 4, 6, 8, 10, 12, None]
linhas_profundidade = {}
for profundidade in profundidades:
    modelo_prof = make_pipeline(criar_prep(grupo_3, False),
                                DecisionTreeClassifier(class_weight='balanced', max_depth=profundidade, random_state=42))
    res_treino, res_validacao = avaliar_treino_validacao(modelo_prof, grupo_3)
    linha = resumir(res_validacao)
    linha['treino (média)'] = pd.Series(res_treino).mean()
    linhas_profundidade[str(profundidade)] = linha

In [ ]:
tabela_profundidade = pd.DataFrame(linhas_profundidade).T
tabela_profundidade[['treino (média)', 'fold 1', 'fold 2', 'fold 3', 'média', 'desvio', 'pior fold', 'folds > regra']]

,treino (média),fold 1,fold 2,fold 3,média,desvio,pior fold,folds > regra
3,0.505336,0.436634,0.567782,0.544225,0.516214,0.069918,0.436634,0.0
4,0.541926,0.468579,0.601635,0.582251,0.550822,0.071881,0.468579,1.0
6,0.576431,0.503096,0.626056,0.606073,0.578408,0.065983,0.503096,3.0
8,0.598528,0.504398,0.636823,0.584592,0.575271,0.066703,0.504398,2.0
10,0.620765,0.484712,0.610586,0.561234,0.552177,0.063424,0.484712,2.0
12,0.652355,0.433126,0.554020,0.522205,0.503117,0.062667,0.433126,0.0
None,1.000000,0.198846,0.273514,0.275533,0.249298,0.043704,0.198846,0.0


## 5.2 Naive Bayes misto (G3)

In [ ]:
# Naive Bayes misto (Aula 4):
class NaiveBayesMisto(ClassifierMixin, BaseEstimator):
    def __init__(self, categoricas, com_log, alpha=1.0):
        self.categoricas = categoricas
        self.com_log = com_log
        self.alpha = alpha

    def _parte_categorica(self, X):
        codigos = self.encoder_.transform(X[self.categoricas])
        # categoria que não apareceu no treino (-1) passa para um código extra,
        # que o CategoricalNB conhece (contagem 0, só o Laplace) e por isso não dá erro
        for j in range(codigos.shape[1]):
            codigos[codigos[:, j] == -1, j] = len(self.encoder_.categories_[j])
        return codigos

    def _parte_numerica(self, X):
        # NaN das colunas "horas desde a última..." = nunca aconteceu -> valor alto (máximo do treino)
        numericas = X[self.numericas_].fillna(self.maximos_)
        for coluna in self.numericas_:
            if coluna in self.com_log:
                numericas[coluna] = np.log1p(numericas[coluna])
        return numericas.values

    def fit(self, X, y):
        self.numericas_ = [c for c in X.columns if c not in self.categoricas]
        self.maximos_ = X[self.numericas_].max()
        # códigos das categorias aprendidos só no treino
        self.encoder_ = OrdinalEncoder(handle_unknown='use_encoded_value', unknown_value=-1)
        self.encoder_.fit(X[self.categoricas])
        n_categorias = [len(c) + 1 for c in self.encoder_.categories_]  # +1 para o código extra
        self.nb_categorico_ = CategoricalNB(alpha=self.alpha, min_categories=n_categorias)
        self.nb_categorico_.fit(self._parte_categorica(X), y)
        self.nb_numerico_ = GaussianNB()
        self.nb_numerico_.fit(self._parte_numerica(X), y)
        self.classes_ = self.nb_categorico_.classes_
        return self

    def predict_proba(self, X):
        # soma das log-verosimilhanças das duas partes; o log do prior vem nas duas, por isso tira-se uma vez
        conjunta = (self.nb_categorico_.predict_joint_log_proba(self._parte_categorica(X))
                    + self.nb_numerico_.predict_joint_log_proba(self._parte_numerica(X))
                    - self.nb_categorico_.class_log_prior_)
        # normalizar para as probabilidades somarem 1
        return np.exp(conjunta - logsumexp(conjunta, axis=1, keepdims=True))

    def predict(self, X):
        return self.classes_[np.argmax(self.predict_proba(X), axis=1)]

In [ ]:
# Colunas do G3 tratadas como categóricas 
# no NB misto (as restantes acabam por ser numéricas)
categoricas_nb = ['Location Information', 'District Name', 'tariff_period', 'hora', 'dia_da_semana',
                  'post_no_previous_completed_session', 'post_no_previous_completed_abnormal']

In [ ]:
# GaussianNB antigo (o que estava na secção 6): one-hot + log1p + escala
modelo_nb_gauss = make_pipeline(criar_prep(grupo_3, True), GaussianNB())
resultados_nb_gauss = avaliar(modelo_nb_gauss, grupo_3)
resultados_nb_gauss

[0.44396281890698674, 0.5862896933577445, 0.5663100062322125]

In [ ]:
# NB misto
modelo_nb_misto = NaiveBayesMisto(categoricas_nb, log_data)
resultados_nb_misto = avaliar(modelo_nb_misto, grupo_3)
resultados_nb_misto

[0.4788136837795993, 0.6011701806730676, 0.5728345796194029]

In [ ]:
# Percentagem de probabilidades extremas (abaixo de 1% ou acima de 99%) na validação do fold 3
prob_gauss = modelo_nb_gauss.predict_proba(validacao_3[grupo_3])[:, 1]
prob_misto = modelo_nb_misto.predict_proba(validacao_3[grupo_3])[:, 1]
extremas_gauss = ((prob_gauss < 0.01) | (prob_gauss > 0.99)).mean()
extremas_misto = ((prob_misto < 0.01) | (prob_misto > 0.99)).mean()

In [ ]:
nb_comparacao = pd.DataFrame({'GaussianNB (atual)': resumir(resultados_nb_gauss), 'NB misto': resumir(resultados_nb_misto)}).T
nb_comparacao['extremas fold 3'] = [extremas_gauss, extremas_misto]
nb_comparacao

,fold 1,fold 2,fold 3,média,desvio,pior fold,folds > regra,extremas fold 3
GaussianNB (atual),0.443963,0.58629,0.566310,0.532188,0.077055,0.443963,0.0,0.767575
NB misto,0.478814,0.60117,0.572835,0.550939,0.064049,0.478814,1.0,0.719870


In [ ]:
# NB misto sem a tariff_period (a tarifa sai da hora: com as duas, a mesma informação conta duas vezes)
grupo_3_sem_tarifa = [c for c in grupo_3 if c != 'tariff_period']
categoricas_nb_sem_tarifa = [c for c in categoricas_nb if c != 'tariff_period']
modelo_nb_sem_tarifa = NaiveBayesMisto(categoricas_nb_sem_tarifa, log_data)
resultados_nb_sem_tarifa = avaliar(modelo_nb_sem_tarifa, grupo_3_sem_tarifa)
resultados_nb_sem_tarifa

[0.47876471458895603, 0.6011167790489205, 0.5728513313718641]

In [ ]:
prob_sem_tarifa = modelo_nb_sem_tarifa.predict_proba(validacao_3[grupo_3_sem_tarifa])[:, 1]
extremas_sem_tarifa = ((prob_sem_tarifa < 0.01) | (prob_sem_tarifa > 0.99)).mean()

nb_tarifa = pd.DataFrame({'NB misto': resumir(resultados_nb_misto), 'NB misto sem tariff_period': resumir(resultados_nb_sem_tarifa)}).T
nb_tarifa['extremas fold 3'] = [extremas_misto, extremas_sem_tarifa]
nb_tarifa

,fold 1,fold 2,fold 3,média,desvio,pior fold,folds > regra,extremas fold 3
NB misto,0.478814,0.601170,0.572835,0.550939,0.064049,0.478814,1.0,0.719870
NB misto sem tariff_period,0.478765,0.601117,0.572851,0.550911,0.064059,0.478765,1.0,0.719492


## 6. Modelos individuais (G3)

In [ ]:
# Regressão logística (linear)
modelo_lr = make_pipeline(criar_prep(grupo_3, True), LogisticRegression(class_weight='balanced', max_iter=1000))
resultados_lr = avaliar(modelo_lr, grupo_3)
resultados_lr

[0.5062081136939464, 0.6364413558112486, 0.6168249626874805]

In [ ]:
# Naive Bayes misto (probabilístico), escolhido na 5.2
modelo_nb = NaiveBayesMisto(categoricas_nb, log_data)
resultados_nb = avaliar(modelo_nb, grupo_3)
resultados_nb

[0.4788136837795993, 0.6011701806730676, 0.5728345796194029]

In [ ]:
# Árvore de decisão, com max_depth=6 escolhido na 5.1
modelo_dt = make_pipeline(criar_prep(grupo_3, False), DecisionTreeClassifier(class_weight='balanced', max_depth=6, random_state=42))
resultados_dt = avaliar(modelo_dt, grupo_3)
resultados_dt

[0.5030962342379672, 0.6260561992098775, 0.6060729477705366]

In [ ]:
individuais = pd.DataFrame({'no-skill': resumir(taxas), 'regra': resumir(regras),
                            'LR': resumir(resultados_lr), 'NB': resumir(resultados_nb), 'DT': resumir(resultados_dt)}).T
individuais

,fold 1,fold 2,fold 3,média,desvio,pior fold,folds > regra
no-skill,0.157910,0.196792,0.210357,0.188353,0.027223,0.157910,0.0
regra,0.450384,0.602265,0.590126,0.547592,0.084403,0.450384,0.0
LR,0.506208,0.636441,0.616825,0.586491,0.070216,0.506208,3.0
NB,0.478814,0.601170,0.572835,0.550939,0.064049,0.478814,1.0
DT,0.503096,0.626056,0.606073,0.578408,0.065983,0.503096,3.0


In [ ]:
# AUC-PR a dividir pela taxa de anomalia de cada fold
individuais[['fold 1', 'fold 2', 'fold 3']] / taxas

,fold 1,fold 2,fold 3
no-skill,1.000000,1.000000,1.000000
regra,2.852162,3.060413,2.805352
LR,3.205682,3.234083,2.932273
NB,3.032200,3.054852,2.723151
DT,3.185975,3.181310,2.881160


In [ ]:
# Coeficientes da regressão logística (modelo treinado no fold 3), os 10 maiores em valor absoluto
coeficientes = pd.Series(modelo_lr[-1].coef_[0], index=modelo_lr[0].get_feature_names_out())
coeficientes.sort_values(key=abs, ascending=False).head(10)

numericas__post_previous_abnormal_rate_1D               0.932391
log__post_previous_abnormal_1D                         -0.804254
categoricas__Location Information_Technology Park      -0.582487
log__post_hours_since_previous_completed_abnormal      -0.572576
numericas__post_previous_abnormal_rate_7D               0.421179
categoricas__Location Information_Tourist Attraction    0.293121
numericas__post_previous_sessions_1D                    0.293031
categoricas__Location Information_Park A                0.227946
categoricas__Location Information_Industrial Park      -0.225785
categoricas__Location Information_Shopping Mall         0.217191
dtype: float64

In [ ]:
# Importância das variáveis na árvore (modelo treinado no fold 3)
importancias = pd.Series(modelo_dt[-1].feature_importances_, index=modelo_dt[0].get_feature_names_out())
importancias.sort_values(ascending=False).head(10)

remainder__post_previous_abnormal_rate_7D                          0.557346
remainder__post_hours_since_previous_completed_abnormal            0.171118
remainder__post_previous_abnormal_rate_1D                          0.148917
remainder__post_previous_abnormal_rate_30D                         0.104494
categoricas__Location Information_Bus Station                      0.003248
remainder__Temperature(℃)                                          0.001943
remainder__post_previous_sessions_1D                               0.001690
categoricas__District Name_Xiuzhou                                 0.001633
categoricas__Location Information_Expressway Service District C    0.001328
categoricas__District Name_Tongxiang                               0.001171
dtype: float64

## 7. Ensembles (G3)

In [ ]:
# Random Forest (bagging): muitas árvores, cada uma treinada numa amostra diferente do treino
modelo_rf = make_pipeline(criar_prep(grupo_3, False), RandomForestClassifier(n_estimators=100, min_samples_leaf=20, class_weight='balanced_subsample', n_jobs=-1, random_state=42))
resultados_rf = avaliar(modelo_rf, grupo_3)
resultados_rf

[0.5332807534296657, 0.6604672782643876, 0.6414911018574234]

In [ ]:
# HistGradientBoosting (boosting): cada árvore corrige os erros das anteriores
modelo_hgb = make_pipeline(criar_prep(grupo_3, False), HistGradientBoostingClassifier(class_weight='balanced', early_stopping=False, random_state=42))
resultados_hgb = avaliar(modelo_hgb, grupo_3)
resultados_hgb

[0.5377748582737041, 0.6607266327853737, 0.641610683433222]

In [ ]:
# Votação soft: média das probabilidades da LR, do NB misto e da DT (cada um com o seu pré-processamento)
modelo_voto = VotingClassifier([('lr', make_pipeline(criar_prep(grupo_3, True), LogisticRegression(class_weight='balanced', max_iter=1000))),
                                ('nb', NaiveBayesMisto(categoricas_nb, log_data)),
                                ('dt', make_pipeline(criar_prep(grupo_3, False), DecisionTreeClassifier(class_weight='balanced', max_depth=6, random_state=42)))],
                               voting='soft')
resultados_voto = avaliar(modelo_voto, grupo_3)
resultados_voto

[0.5135352912244381, 0.6434514566167363, 0.6220912937385027]

In [ ]:
# Percentagem de probabilidades extremas (abaixo de 1% ou acima de 99%) na validação do fold 3
prob_nb = modelo_nb.predict_proba(validacao_3[grupo_3])[:, 1]
prob_lr = modelo_lr.predict_proba(validacao_3[grupo_3])[:, 1]
print('NB:', ((prob_nb < 0.01) | (prob_nb > 0.99)).mean())
print('LR:', ((prob_lr < 0.01) | (prob_lr > 0.99)).mean())

NB: 0.7198700145477904
LR: 0.0029662377902473125


In [ ]:
# Votação soft sem o NB, para ver se ele está a atrapalhar
modelo_voto_sem_nb = VotingClassifier([('lr', make_pipeline(criar_prep(grupo_3, True), LogisticRegression(class_weight='balanced', max_iter=1000))),
                                       ('dt', make_pipeline(criar_prep(grupo_3, False), DecisionTreeClassifier(class_weight='balanced', max_depth=6, random_state=42)))],
                                      voting='soft')
resultados_voto_sem_nb = avaliar(modelo_voto_sem_nb, grupo_3)
resultados_voto_sem_nb

[0.5237498006641461, 0.6500362027735701, 0.6280607108678153]

In [ ]:
ensembles = pd.DataFrame({'regra': resumir(regras), 'LR': resumir(resultados_lr), 'NB': resumir(resultados_nb), 'DT': resumir(resultados_dt),
                          'RF': resumir(resultados_rf), 'HistGB': resumir(resultados_hgb),
                          'Votação': resumir(resultados_voto), 'Votação sem NB': resumir(resultados_voto_sem_nb)}).T
ensembles

,fold 1,fold 2,fold 3,média,desvio,pior fold,folds > regra
regra,0.450384,0.602265,0.590126,0.547592,0.084403,0.450384,0.0
LR,0.506208,0.636441,0.616825,0.586491,0.070216,0.506208,3.0
NB,0.478814,0.601170,0.572835,0.550939,0.064049,0.478814,1.0
DT,0.503096,0.626056,0.606073,0.578408,0.065983,0.503096,3.0
RF,0.533281,0.660467,0.641491,0.611746,0.068612,0.533281,3.0
HistGB,0.537775,0.660727,0.641611,0.613371,0.066162,0.537775,3.0
Votação,0.513535,0.643451,0.622091,0.593026,0.069665,0.513535,3.0
Votação sem NB,0.523750,0.650036,0.628061,0.600616,0.067468,0.523750,3.0


In [ ]:
# AUC-PR a dividir pela taxa de anomalia de cada fold
ensembles[['fold 1', 'fold 2', 'fold 3']] / taxas

,fold 1,fold 2,fold 3
regra,2.852162,3.060413,2.805352
LR,3.205682,3.234083,2.932273
NB,3.032200,3.054852,2.723151
DT,3.185975,3.181310,2.881160
RF,3.377126,3.356171,3.049531
HistGB,3.405586,3.357488,3.050100
Votação,3.252083,3.269705,2.957308
Votação sem NB,3.316769,3.303165,2.985686


## 8. Charging Post ID e contas de frota

O UserID não é testado, por questões de privacidade do usuário.

In [ ]:
# O Charging Post ID é um código e não um número: entra como categoria (one-hot)
colunas_categoricas = colunas_categoricas + ['Charging Post ID']
colunas_categoricas

['Location Information', 'District Name', 'tariff_period', 'Charging Post ID']

In [ ]:
# G3 + Charging Post ID
grupo_3_posto = grupo_3 + ['Charging Post ID']
len(grupo_3_posto)

21

In [ ]:
# HistGB (o melhor modelo da etapa 7) no G3 + Charging Post ID
modelo_hgb_posto = make_pipeline(criar_prep(grupo_3_posto, False), HistGradientBoostingClassifier(class_weight='balanced', early_stopping=False, random_state=42))
resultados_hgb_posto = avaliar(modelo_hgb_posto, grupo_3_posto)
resultados_hgb_posto

[0.537838569577973, 0.6606710337144055, 0.6392056867427386]

In [ ]:
pd.DataFrame({'HistGB G3': resumir(resultados_hgb), 'HistGB G3 + Charging Post ID': resumir(resultados_hgb_posto)}).T

,fold 1,fold 2,fold 3,média,desvio,pior fold,folds > regra
HistGB G3,0.537775,0.660727,0.641611,0.613371,0.066162,0.537775,3.0
HistGB G3 + Charging Post ID,0.537839,0.660671,0.639206,0.612572,0.065605,0.537839,3.0


In [ ]:
# Contas de frota identificadas na Task 1
frota = [39, 23]

In [ ]:
# AUC-PR do HistGB (G3) em todas as sessões, sem as sessões de frota e só nas sessões de frota
# só avaliação: o modelo é o mesmo, treinado com todas as sessões do treino de cada fold
com_frota = []
sem_frota = []
frota_only = []
for treino, validacao in folds:
    modelo_hgb.fit(treino[grupo_3], treino['is_Abnormal'])
    prob = modelo_hgb.predict_proba(validacao[grupo_3])[:, 1]
    e_frota = validacao['UserID'].isin(frota).values
    com_frota.append(average_precision_score(validacao['is_Abnormal'], prob))
    sem_frota.append(average_precision_score(validacao['is_Abnormal'][~e_frota], prob[~e_frota]))
    frota_only.append(average_precision_score(validacao['is_Abnormal'][e_frota], prob[e_frota]))

pd.DataFrame({'com frota': com_frota, 'sem frota': sem_frota, 'só frota': frota_only}, index=['fold 1', 'fold 2', 'fold 3'])

,com frota,sem frota,só frota
fold 1,0.537775,0.539256,0.503817
fold 2,0.660727,0.659673,0.684592
fold 3,0.641611,0.642716,0.617699


In [ ]:
# Taxa de anomalia das sessões de frota em cada validação
[validacao_1[validacao_1['UserID'].isin(frota)]['is_Abnormal'].mean(),
 validacao_2[validacao_2['UserID'].isin(frota)]['is_Abnormal'].mean(),
 validacao_3[validacao_3['UserID'].isin(frota)]['is_Abnormal'].mean()]

[np.float64(0.139723414401526),
 np.float64(0.1991701244813278),
 np.float64(0.2184602982668279)]

## 9. Comparação final

In [ ]:
# Todos os modelos no G3, ordenados pela média da AUC-PR
comparacao = pd.DataFrame({'no-skill': resumir(taxas), 'regra': resumir(regras),
                           'LR': resumir(resultados_lr), 'NB': resumir(resultados_nb), 'DT': resumir(resultados_dt),
                           'RF': resumir(resultados_rf), 'HistGB': resumir(resultados_hgb),
                           'Votação': resumir(resultados_voto), 'Votação sem NB': resumir(resultados_voto_sem_nb)}).T
comparacao.sort_values('média')

,fold 1,fold 2,fold 3,média,desvio,pior fold,folds > regra
no-skill,0.157910,0.196792,0.210357,0.188353,0.027223,0.157910,0.0
regra,0.450384,0.602265,0.590126,0.547592,0.084403,0.450384,0.0
NB,0.478814,0.601170,0.572835,0.550939,0.064049,0.478814,1.0
DT,0.503096,0.626056,0.606073,0.578408,0.065983,0.503096,3.0
LR,0.506208,0.636441,0.616825,0.586491,0.070216,0.506208,3.0
Votação,0.513535,0.643451,0.622091,0.593026,0.069665,0.513535,3.0
Votação sem NB,0.523750,0.650036,0.628061,0.600616,0.067468,0.523750,3.0
RF,0.533281,0.660467,0.641491,0.611746,0.068612,0.533281,3.0
HistGB,0.537775,0.660727,0.641611,0.613371,0.066162,0.537775,3.0


In [ ]:
# Diferença para a regra em cada fold (positivo = melhor que a regra)
(comparacao[['fold 1', 'fold 2', 'fold 3']] - regras).sort_values('fold 1')

,fold 1,fold 2,fold 3
no-skill,-0.292474,-0.405473,-0.379769
regra,0.000000,0.000000,0.000000
NB,0.028430,-0.001094,-0.017292
DT,0.052712,0.023792,0.015947
LR,0.055824,0.034177,0.026699
Votação,0.063151,0.041187,0.031965
Votação sem NB,0.073366,0.047772,0.037934
RF,0.082897,0.058203,0.051365
HistGB,0.087391,0.058462,0.051484


Candidatos para a etapa seguinte: HistGB (melhor média, melhor pior fold e menor desvio) e Random Forest.

## Anexo: verificações e testes

### Secção 1 (preparação)

In [ ]:
len(test) / len(df)

0.20000136030670382

In [ ]:
test['Start Time'].min()

Timestamp('2021-09-22 18:55:31')

In [ ]:
validacao_1['is_Abnormal'].sum()

np.int64(8358)

In [ ]:
validacao_2['is_Abnormal'].sum()

np.int64(10416)

In [ ]:
validacao_3['is_Abnormal'].sum()

np.int64(11134)

In [ ]:
validacao_2['is_Abnormal'].mean()

np.float64(0.19679192881029303)

In [ ]:
validacao_3['is_Abnormal'].mean()

np.float64(0.21035727106123298)

### Secção 2 (grupos de variáveis)

In [ ]:
# Colunas do df que não entram em nenhum grupo
df.columns.difference(grupo_4)

Index(['Charging Post ID', 'Start Time', 'UserID', 'is_Abnormal'], dtype='str')

### Secção 3 (pré-processamento)

In [ ]:
df[log_data].skew()

post_hours_since_previous_completed_session     109.628185
post_hours_since_previous_completed_abnormal     11.435940
post_previous_abnormal_1D                         4.748421
post_previous_abnormal_7D                         3.998407
post_previous_abnormal_30D                        3.075707
user_previous_sessions_30D                        7.719969
user_previous_sessions_180D                       5.724352
user_previous_sessions_730D                       6.233589
user_days_since_previous_completed_session        8.179763
user_days_since_previous_completed_abnormal       4.350628
user_active_sessions_at_start                     8.994392
dtype: float64

In [ ]:
# Os NaN estruturais (Task 1) estão todos dentro das log_data
x=df[grupo_4].isna().sum().sum() 
y=df[log_data].isna().sum().sum()
print(x==y)

True


In [ ]:
# Teste no fold 1: versão linear do G4, ajustada só no treino
prep_linear = criar_prep(grupo_4, True)
X_linear = prep_linear.fit_transform(treino_1[grupo_4])
X_linear.shape

(180000, 79)

In [ ]:
# A regressão logística e o Naive Bayes não aceitam NaN: tem de dar 0
np.isnan(X_linear).sum()

np.int64(0)

In [ ]:
# Versão das árvores do G4
prep_arvore = criar_prep(grupo_4, False)
X_arvore = prep_arvore.fit_transform(treino_1[grupo_4])
X_arvore.shape

(180000, 50)

In [ ]:
# Nas árvores os NaN continuam lá (são suportados)
np.isnan(X_arvore).sum()

np.int64(70002)

In [ ]:
# O G1 não tem colunas de histórico e mesmo assim funciona
criar_prep(grupo_1, True).fit_transform(treino_1[grupo_1]).shape

(180000, 50)

### Secção 4 (função de avaliação)

In [ ]:
# Teste: regressão logística no G1
modelo_teste = make_pipeline(criar_prep(grupo_1, True), LogisticRegression(class_weight='balanced', max_iter=1000))
resultados_teste = avaliar(modelo_teste, grupo_1)
resultados_teste

[0.24902410177233109, 0.31011679350546995, 0.33449125149561815]

In [ ]:
pd.DataFrame({'no-skill': resumir(taxas), 'regra': resumir(regras), 'LR G1 (teste)': resumir(resultados_teste)}).T

,fold 1,fold 2,fold 3,média,desvio,pior fold,folds > regra
no-skill,0.157910,0.196792,0.210357,0.188353,0.027223,0.157910,0.0
regra,0.450384,0.602265,0.590126,0.547592,0.084403,0.450384,0.0
LR G1 (teste),0.249024,0.310117,0.334491,0.297877,0.044029,0.249024,0.0


### Secção 5.2 (Naive Bayes misto)

In [ ]:
# Versão do scikit-learn e métodos usados no NB misto
print(sklearn.__version__)
print('CategoricalNB.predict_joint_log_proba:', hasattr(CategoricalNB, 'predict_joint_log_proba'))
print('GaussianNB.predict_joint_log_proba:', hasattr(GaussianNB, 'predict_joint_log_proba'))

1.9.1
CategoricalNB.predict_joint_log_proba: True
GaussianNB.predict_joint_log_proba: True


In [ ]:
# Teste no fold 1: categorias que não existem no treino não podem dar erro
nb_teste = NaiveBayesMisto(categoricas_nb, log_data).fit(treino_1[grupo_3], treino_1['is_Abnormal'])
X_novo = validacao_1[grupo_3].head(5).copy()
X_novo['Location Information'] = 'Posto novo'
X_novo['hora'] = 25
prob_novo = nb_teste.predict_proba(X_novo)
print(prob_novo)
print('somam 1:', np.allclose(prob_novo.sum(axis=1), 1))

[[7.45121502e-01 2.54878498e-01]
 [8.06070507e-01 1.93929493e-01]
 [9.76241150e-01 2.37588498e-02]
 [4.30926712e-04 9.99569073e-01]
 [9.93435201e-01 6.56479945e-03]]
somam 1: True


In [ ]:
# O prior é o mesmo nas duas partes (por isso basta tirá-lo uma vez)
np.allclose(np.log(nb_teste.nb_numerico_.class_prior_), nb_teste.nb_categorico_.class_log_prior_)

True

In [ ]:
# NaN nas colunas do G3, no treino e na validação de cada fold
for treino, validacao in folds:
    print(treino[grupo_3].isna().sum().sum(), validacao[grupo_3].isna().sum().sum())

0 0
0 0
0 0


# Task 4 - Optimização de hiperparâmetros e validação cruzada

In [ ]:
from sklearn.base import clone
from sklearn.metrics import f1_score, precision_score, recall_score
from sklearn.model_selection import GridSearchCV

In [7]:
def dividir_treino_teste(df, prop_treino=0.8):
    corte = int(len(df) * prop_treino)
    treino = df.iloc[:corte]
    teste = df.iloc[corte:]
    return treino, teste

In [6]:
def dividir_folds(treino, n_folds, prop_validacao, tamanho_treino):
    tamanho_validacao = int(len(treino) * prop_validacao / n_folds)
    tss = TimeSeriesSplit(n_splits=n_folds,
                          test_size=tamanho_validacao,
                          max_train_size=tamanho_treino)

    folds = []
    for indices_treino, indices_validacao in tss.split(treino):
        folds.append((treino.iloc[indices_treino], treino.iloc[indices_validacao]))
    return folds

In [ ]:
# Para o GridSearchCV, que não aceita TimeSeriesSplit com max_train_size, vamos criar um TimeSeriesSplit "manual" com os mesmos cortes do tss acima

prop_validacao = 0.45
tamanho_treino = 180000
n_folds = 3
tamanho_validacao = int(len(treino) * prop_validacao / n_folds)
tss = TimeSeriesSplit(n_splits=n_folds,
                          test_size=tamanho_validacao,
                          max_train_size=tamanho_treino)

In [ ]:
param_grid_rf = {
    'randomforestclassifier__n_estimators': [100, 200, 300, 400, 500],
    'randomforestclassifier__max_depth': [None, 10, 20, 30],
    'randomforestclassifier__min_samples_split': [2, 5, 10],
    'randomforestclassifier__max_features': [1.0, 'sqrt', 'log2']
}

modelo_rf_tuning = make_pipeline(
    criar_prep(grupo_3, False),
    RandomForestClassifier(
        class_weight='balanced_subsample',
        n_jobs=-1,
        random_state=42
    )
)

grid_rf = GridSearchCV(
    estimator=modelo_rf_tuning,
    param_grid=param_grid_rf,
    scoring='f1',  # AUC-PR não é diretamente suportada, mas f1 é uma métrica de classificação binária
    cv=tss,
    n_jobs=-1,
    refit=True,
    verbose=1
)

grid_rf.fit(df[grupo_3], df['is_Abnormal'])

print('Melhores parâmetros:')
print(grid_rf.best_params_)
print('Melhor pontuação (f1):', grid_rf.best_score_)

In [ ]:
treino, teste = dividir_treino_teste(df, prop_treino=0.8)

X_treino = treino[grupo_3]
y_treino = treino['is_Abnormal']

# Hiperparâmetros selecionados pelo GridSearchCV
melhor_modelo_rf = clone(grid_rf.best_estimator_)

y_true_oof_rf = []
y_prob_oof_rf = []

for train_idx, valid_idx in tss.split(X_treino):
    model = clone(melhor_modelo_rf)

    model.fit(X_treino.iloc[train_idx], y_treino.iloc[train_idx])

    prob = model.predict_proba(X_treino.iloc[valid_idx])[:, 1]

    y_true_oof_rf.extend(y_treino.iloc[valid_idx].to_numpy())
    y_prob_oof_rf.extend(prob)

y_true_oof_rf = np.asarray(y_true_oof_rf)
y_prob_oof_rf = np.asarray(y_prob_oof_rf)

# Procurar o threshold que maximiza o F1-score
thresholds = np.arange(0.05, 0.96, 0.01)

scores = [
    f1_score(
        y_true_oof_rf,
        (y_prob_oof_rf >= threshold).astype(int),
        zero_division=0
    )
    for threshold in thresholds
]

melhor_idx_rf = np.argmax(scores)

melhor_threshold_rf = thresholds[melhor_idx_rf]
melhor_f1_rf = scores[melhor_idx_rf]

print("Melhor threshold:", melhor_threshold_rf)
print("F1-score temporal OOF:", melhor_f1_rf)

In [ ]:
param_grid_hgb = {
    'histgradientboostingclassifier__max_iter': [100, 200, 300],
    'histgradientboostingclassifier__max_depth': [None, 10, 20],
    'histgradientboostingclassifier__learning_rate': [0.01, 0.1, 0.1],
    'histgradientboostingclassifier__min_samples_leaf': [20, 50, 100]
}

pipeline_hgb_tuning = make_pipeline(
    criar_prep(grupo_3, False),
    HistGradientBoostingClassifier(
        class_weight='balanced',
        early_stopping=False,
        random_state=42
    )
)

grid_hgb = GridSearchCV(
    estimator=pipeline_hgb_tuning,
    param_grid=param_grid_hgb,
    scoring='f1',  # AUC-PR não é diretamente suportada, mas f1 é uma métrica de classificação binária
    cv=tss,
    n_jobs=-1,
    refit=True,
    verbose=1
)

grid_hgb.fit(df[grupo_3], df['is_Abnormal'])

print('Melhores parâmetros:')
print(grid_hgb.best_params_)
print('Melhor pontuação (f1):', grid_hgb.best_score_)

In [ ]:
treino, teste = dividir_treino_teste(df, prop_treino=0.8)

X_treino = treino[grupo_3]
y_treino = treino['is_Abnormal']

best_model_hgb = clone(grid_hgb.best_estimator_)

y_true_oof_hgb = []
y_prob_oof_hgb = []

for train_idx, valid_idx in tss.split(X_treino):
    model_hgb = clone(best_model_hgb)

    model_hgb.fit(X_treino.iloc[train_idx], y_treino.iloc[train_idx])

    prob_hgb = model_hgb.predict_proba(X_treino.iloc[valid_idx])[:, 1]

    y_true_oof_hgb.extend(y_treino.iloc[valid_idx].to_numpy())
    y_prob_oof_hgb.extend(prob_hgb)

y_true_oof_hgb = np.asarray(y_true_oof_hgb)
y_prob_oof_hgb = np.asarray(y_prob_oof_hgb)

# Procurar o threshold que maximiza o F1-score
thresholds_hgb = np.arange(0.05, 0.96, 0.01)

scores_hgb = [
    f1_score(
        y_true_oof_hgb,
        (y_prob_oof_hgb >= threshold).astype(int),
        zero_division=0
    )
    for threshold in thresholds_hgb
]

melhor_idx_hgb = np.argmax(scores_hgb)
melhor_threshold_hgb = thresholds_hgb[melhor_idx_hgb]
melhor_f1_hgb = scores_hgb[melhor_idx_hgb]

print("Melhor threshold HGB:", melhor_threshold_hgb)
print("F1-score temporal OOF HGB:", melhor_f1_hgb)

In [ ]:
def avaliar_modelo(y_true, y_pred):
    return {
        "Precision (TP / (TP + FP))": precision_score(y_true, y_pred, zero_division=0),
        "Recall (TP / (TP + FN))": recall_score(y_true, y_pred, zero_division=0),
        "F1-Score (2 * (Precision * Recall) / (Precision + Recall))": f1_score(y_true, y_pred, zero_division=0),
        "Alert rate (Number of alerts / Total sessions)": np.mean(y_pred),
        "Correct alerts per 1,000 sessions (TP)": np.sum((y_true == 1) & (y_pred == 1)) / len(y_true) * 1000,
        "False alerts per 1,000 sessions (FP)": np.sum((y_true == 0) & (y_pred == 1)) / len(y_true) * 1000,
        "Missed abnormal sessions per 1,000 sessions (FN)": np.sum((y_true == 1) & (y_pred == 0)) / len(y_true) * 1000
    }

In [ ]:
## Avaliação do modelo Random Forest
X_test = teste[grupo_3]
y_test = teste['is_Abnormal']

# Treinar o melhor modelo Random Forest no conjunto de treino completo
modelo_final_rf = clone(grid_rf.best_estimator_)
modelo_final_rf.fit(X_treino, y_treino)

# Previsões no conjunto de teste
y_prob_test_rf = modelo_final_rf.predict_proba(X_test)[:, 1]

# Aplicar o threshold otimizado
y_pred_test_rf = (y_prob_test_rf >= melhor_threshold_rf).astype(int)

# Calcular métricas
metrics_rf = avaliar_modelo(y_test, y_pred_test_rf)

print("Métricas do modelo Random Forest no conjunto de teste:")
metricas_rf_df = pd.DataFrame(metrics_rf, index=[0])
metricas_rf_df

In [ ]:
## Avaliação do modelo HistGradientBoosting
X_test = test[grupo_3]
y_test = test['is_Abnormal']

# Treinar o melhor modelo com todos os dados de treino
modelo_final_hgb = clone(grid_hgb.best_estimator_)
modelo_final_hgb.fit(X_treino, y_treino)

# Probabilidades no conjunto de teste
y_prob_hgb = modelo_final_hgb.predict_proba(X_test)[:, 1]

# Aplicar o threshold otimizado
y_pred_hgb = (y_prob_hgb >= melhor_threshold_hgb).astype(int)

# Calcular métricas
metrics_hgb = avaliar_modelo(y_test, y_pred_hgb)

print("Métricas do modelo HistGradientBoosting:")
metrics_hgb_df = pd.DataFrame(metrics_hgb, index=[0])
metrics_hgb_df